# Scraping Statistics Analysis

This notebook dynamically computes and presents the key empirical statistics of the scraped arXiv corpus, as required by the Milestone 1 statement.

All metrics and plots are generated **during Jupyter Book compilation** from `data/papers.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Locate data file: works whether built locally in docs/ or from repository root
data_candidates = [
    Path("../data/papers.csv"),
    Path("./data/papers.csv"),
    Path("../../logs/papers.csv"),
    Path("../logs/papers.csv"),
]
csv_path = next((p for p in data_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Cannot locate papers.csv in standard locations.")

df = pd.read_csv(csv_path)
numeric_cols = ["versions", "versions_ok", "size_before", "size_after", "n_refs", "n_arxiv_refs", "seconds"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print(f"Loaded {len(df)} paper records from {csv_path}")


## 1. Summary Metrics Table

In [ ]:
total_papers = len(df)
ok_papers = df[df["status"] == "ok"]
n_ok = len(ok_papers)
success_rate = (n_ok / total_papers * 100) if total_papers > 0 else 0.0

downloaded = df[df["size_before"] > 0]
avg_size_before = downloaded["size_before"].mean() if len(downloaded) > 0 else 0.0
avg_size_after = downloaded["size_after"].mean() if len(downloaded) > 0 else 0.0
saved_pct = ((1.0 - (avg_size_after / avg_size_before)) * 100) if avg_size_before > 0 else 0.0

ref_checked = df[df["status"].isin(["ok", "partial"])]
avg_refs = ref_checked["n_refs"].mean() if len(ref_checked) > 0 else 0.0
avg_arxiv_refs = ref_checked["n_arxiv_refs"].mean() if len(ref_checked) > 0 else 0.0

summary_data = [
    ("Total Papers Processed", total_papers),
    ("Successfully Crawled Papers", n_ok),
    ("Overall Crawl Success Rate", f"{success_rate:.2f}%"),
    ("Avg Size Before Figure Removal", f"{avg_size_before:,.0f} Bytes ({avg_size_before / 1024:.1f} KB)"),
    ("Avg Size After Figure Removal", f"{avg_size_after:,.0f} Bytes ({avg_size_after / 1024:.1f} KB)"),
    ("Storage Space Saved", f"{saved_pct:.1f}%"),
    ("Avg References per Paper (Total)", f"{avg_refs:.2f}"),
    ("Avg References per Paper (arXiv Only)", f"{avg_arxiv_refs:.2f}"),
    ("Avg Processing Time per Paper", f"{df['seconds'].mean():.2f} seconds"),
]

summary_table = pd.DataFrame(summary_data, columns=["Metric", "Value"])
summary_table


## 2. Average Paper Size: Before vs. After Figure Pruning

Extracting only `.tex` and `.bib` files and pruning binary figures drastically conserves disk space on CPU execution runners.

In [ ]:
avg_before_kb = avg_size_before / 1024
avg_after_kb = avg_size_after / 1024

fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(["Before Removal", "After Removal"], [avg_before_kb, avg_after_kb],
              color=["#d9534f", "#5cb85c"], width=0.45)

for bar in bars:
    h = bar.get_height()
    ax.annotate(f"{h:.1f} KB",
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontweight="bold")

ax.set_title(f"Average Paper Size Reduction (Space Saved: {saved_pct:.1f}%)", fontsize=12, fontweight="bold")
ax.set_ylabel("Size (KB)", fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 3. Paper Revisions Distribution

Shows the breakdown of papers by their total revision count on arXiv.

In [ ]:
counts = df["versions"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([f"{v} version(s)" for v in counts.index], counts.values, color="#45818e", width=0.5)

for bar in bars:
    h = bar.get_height()
    ax.annotate(f"{h}",
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom")

ax.set_title("Distribution of Paper Revisions", fontsize=12, fontweight="bold")
ax.set_ylabel("Number of Papers", fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 4. Citation Graph Density

Distribution of total references retrieved via Semantic Scholar versus references holding valid arXiv identifiers.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["n_refs"], bins=10, alpha=0.7, color="#337ab7", label="All Outbound References (Semantic Scholar)")
ax.hist(df["n_arxiv_refs"], bins=10, alpha=0.7, color="#6f5499", label="References with arXiv IDs")

ax.set_title("Distribution of References per Paper", fontsize=12, fontweight="bold")
ax.set_xlabel("Reference Count", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
